# Keep or go: a store's directories and its handwritten skills, judged without the owner's history

A Claude Code session reads CLAUDE.md, the auto-memory and p4-recall's lines before it answers, so its call on which
directories to keep leans on how the owner has worked. This notebook asks the Claude API directly: bare Messages
requests, a neutral system prompt, evidence computed from git (never written by an agent), owner names scrubbed, the
unit order shuffled per seed, and two model families judging. Step 3 measures what lean is left: the same evidence
unscrubbed, and every verdict that moves.

Then each handwritten skill is split into rules (GENERIC / REPO_SPECIFIC / OBSOLETE) with a generic SKILL.md drafted
from them, and probe tasks are answered with the handwritten skill, the generic draft and no skill: the draft can
replace the handwritten one in the p4 marketplace when it scores like the handwritten arm and above `none`.

`FAKE = True` (the default) runs every cell offline, with no key and no cost. The paid run is `FAKE = False` here, or
`python experiments/keep_or_go/keep_or_go.py run --store <store> --skill <dir> --paid` (the project's `.venv`, and `ANTHROPIC_API_KEY` in `.env` or
`ant auth login`). It stops before any request once `TOTAL_MAX_USD` ($6) is spent; one run is
about $1.50 to $3 at the defaults.

## Settings

In [ ]:
import os, sys, json, pathlib
HERE = pathlib.Path(os.environ.get("KEEP_OR_GO_DIR", ".")).resolve()   # this folder, inside it, or the project root
DIR = next((p for p in [HERE, *HERE.parents, HERE / "experiments" / "keep_or_go"] if (p / "keep_or_go.py").exists()),
           None)
sys.path.insert(0, str(DIR))
import keep_or_go as kg

FAKE = True                                   # False: the paid run
TOTAL_MAX_USD = 6.0
STORE_REPO = os.environ.get("P4_STORE_REPO", "")   # required: a p4 store repository, a URL or a local path
BRANCHES = []                                 # [] = every remote branch
SKILL_DIRS = [d for d in os.environ.get("P4_SKILL_DIRS", "").split(os.pathsep) if d]   # handwritten skills; [] skips 4-5
PLUGIN_REPO = os.environ.get("P4_PROJECT", "")   # the repository the marketplace skills' settings read
JUDGE_MODELS = ['claude-opus-5-5', 'claude-sonnet-5-5']
SEEDS = [1, 2]
PROBE_MODEL, PROBE_JUDGE = 'claude-opus-5-5', 'claude-sonnet-5-5'
WORK = os.environ.get("KEEP_OR_GO_WORK", str(kg.WORK))
kg.configure(FAKE=FAKE, TOTAL_MAX_USD=TOTAL_MAX_USD, STORE_REPO=STORE_REPO, BRANCHES=BRANCHES, SKILL_DIRS=SKILL_DIRS, PLUGIN_REPO=PLUGIN_REPO,
             JUDGE_MODELS=JUDGE_MODELS, SEEDS=SEEDS, PROBE_MODEL=PROBE_MODEL, PROBE_JUDGE=PROBE_JUDGE, WORK=WORK)
print("FAKE" if FAKE else "PAID", "| cap $", TOTAL_MAX_USD, "| work", WORK)

## 1. Inventory (free)
One unit per directory (a directory with more than 12 children is one unit), per branch: files, bytes, commits, last touch, the share of blobs identical on every branch, and which code repositories name its path.

In [ ]:
inv = kg.step_inventory()
for u in inv:
    print(f"{u['unit']:40} {sorted(u['branches'])} same={u['identical_share']:.0%} refs={u['referenced_by']}")

The exact text every judge receives (seed 1's order):

In [ ]:
print(kg.evidence(inv, True, SEEDS[0])[:4000])

## 2. Judge (paid)

In [ ]:
res = kg.step_judge(inv)
for t in kg.consensus(res):
    print(f"{t['verdict']:8} {t['agreement']:.0%}  {t['unit']}")

## 3. Lean check (paid)
The first judge and seed with the owner's wording left in.

In [ ]:
moved = kg.step_ablate(inv, res)
moved

## 4. Skill rules and generic drafts (paid)

In [ ]:
rules = kg.step_rules()
for n, r in rules.items():
    print(n)
    for x in r['rules']:
        print(f"  {x['class']:13} {x['rule'][:100]}")

## 5. Probes: handwritten vs generic vs none (paid)

In [ ]:
probe_rows, probe_summary = kg.step_probe(rules)
probe_summary

## 6. Report

In [ ]:
summary = kg.step_report(inv, res, moved, rules, probe_summary)
print((kg.paths()['results'] / 'report.md').read_text(encoding='utf-8'))